# Step 3: your diagram applied to the pretrained RS-Diffusion (no training)
RS-Diffusion already predicts a **64x64 flow**, upsamples and rescales it, and warps the **full-resolution image**. This notebook keeps the pretrained model and the *same predicted flow* for every sample, and only changes what happens **after** the flow is predicted, so each box of the diagram is tested on its own:

| Variant | What it tests |
|---|---|
| A. low-res warp + image upscale | the naive alternative: warp the 64x64 image, then upscale the corrected image |
| **B. full-res warp, flow bilinear + rescale** | **the diagram's design (= the Step 1 baseline)** |
| C. flow upsampled bicubic | does the interpolation mode of the "upsample flow" box matter? |
| D. flow upsampled nearest | same, crude interpolation |
| E. bilinear, **no value rescale** | is the "rescale" half of "resize + rescale" really needed? |

Each variant is scored at **256x256** (npz GS) and at **800x600** (the dataset's jpg GS, showing the benefit of full-resolution warping), for RS-Diffusion at 8 and 1 DDIM steps. An optional last cell runs the diffusion model itself at a higher input size (no retraining) to test the "cheap low-res compute" claim.

**Settings:** Accelerator GPU (T4), Internet **On**. Nothing is trained. Outputs: `/kaggle/working/flowdesign_results.csv`.

## 1. Setup and settings

In [1]:
import os, sys, glob, yaml, time, gc
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch, torch.nn.functional as F
from torchvision import transforms as T
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU, then restart."
!pip install -q huggingface_hub einops accelerate ema-pytorch pyyaml tqdm opencv-python-headless timm denoising-diffusion-pytorch scikit-image

if not os.path.exists('/kaggle/working/RS-Diffusion'):
    !git clone https://github.com/lhaippp/RS-Diffusion /kaggle/working/RS-Diffusion
REPO = '/kaggle/working/RS-Diffusion'
assert os.path.isdir(f'{REPO}/model'), "git clone failed: turn Internet ON in the notebook settings and re-run."
%cd /kaggle/working/RS-Diffusion
sys.path.insert(0, REPO)

# ---------------- settings ----------------
N_REAL       = 50            # same 50 test samples as Steps 1 and 2
STEPS_LIST   = [8, 1]        # DDIM steps of the pretrained model (8 = config baseline)
N_WARMUP     = 3
RUN_HIGHRES  = True          # optional: run the diffusion model itself at a larger input size (no retraining)
HIGHRES      = [128]
CKPT_REPO, CKPT_FILE = 'Yzl-code/RS-Diffusion', 'RS_Real.pt'
# -------------------------------------------
ckpt_path = f'{REPO}/checkpoint/{CKPT_FILE}'
os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
from huggingface_hub import hf_hub_download, HfApi
if not os.path.isfile(ckpt_path):
    hf_hub_download(repo_id=CKPT_REPO, repo_type='dataset', filename=CKPT_FILE, local_dir=f'{REPO}/checkpoint')

DATA_DIR = '/kaggle/temp/RS-Real'
files = sorted(f for f in HfApi().list_repo_files(CKPT_REPO, repo_type='dataset') if f.startswith('test/npz/') and f.endswith('.npz'))
pick = [files[i] for i in np.linspace(0, len(files) - 1, min(N_REAL, len(files))).astype(int)]
def fetch(fs):
    def one(f):
        if not os.path.isfile(f'{DATA_DIR}/{f}'): hf_hub_download(repo_id=CKPT_REPO, repo_type='dataset', filename=f, local_dir=DATA_DIR)
    with ThreadPoolExecutor(16) as ex: list(tqdm(ex.map(one, fs), total=len(fs)))
fetch(pick); fetch([f.replace('/npz/', '/jpg/').replace('.npz', '.jpg') for f in pick])     # npz for the model, jpg for the 800x600 evaluation
real_paths = [f'{DATA_DIR}/{f}' for f in pick]
print(len(real_paths), 'test samples (npz + jpg)')

Cloning into '/kaggle/working/RS-Diffusion'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 87 (delta 43), reused 39 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 43.79 KiB | 7.30 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/kaggle/working/RS-Diffusion


RS_Real.pt: reconstructing file:   0%|          |  0.00B / 1.28GB            

RS_Real.pt: downloading bytes:           |  0.00B            

  0%|          | 0/50 [00:00<?, ?it/s]

test/npz/scene03_0771.npz: reconstructing file:   0%|          |  0.00B /  918kB            

test/npz/scene04_0893.npz: reconstructing file:   0%|          |  0.00B /  918kB            

... (196 outputs left out)

test/jpg/scene26_0427.jpg: downloading bytes:           |  0.00B            

test/jpg/scene25_0717.jpg: downloading bytes:           |  0.00B            

test/jpg/scene26_1037.jpg: downloading bytes:           |  0.00B            

50 test samples (npz + jpg)


## 2. Data and model

In [2]:
from dataset.RS_Real_dataset import RS_Real_test_dataset
from model.diffusion_model import Unet, RS_Diffusion
from flow_utils import flow_warp, upsample2d_flow_as
cfg = yaml.safe_load(open(f'{REPO}/config/RS_real_config_teat.yaml'))
ds = RS_Real_test_dataset(folder=os.path.dirname(real_paths[0]), image_size=cfg['dataset']['image_size'], augment_horizontal_flip=False)
tf = ds.transform
to_t = T.ToTensor()

def load_samples(paths):
    out = []
    for p in paths:
        d = np.load(p)
        im = Image.open(p.replace('/npz/', '/jpg/').replace('.npz', '.jpg')).convert('RGB'); w, h = im.size   # 1600x600: left = RS, right = GS
        out.append(dict(name=Path(p).stem, flow=torch.from_numpy(d['out_flow']).float()[None],
                        rs=tf(Image.fromarray(d['condition']))[None],
                        rs_hi=torch.from_numpy(d['condition'].copy()).permute(2, 0, 1).float()[None] / 255,
                        gs_hi=torch.from_numpy(d['gs'].copy()).permute(2, 0, 1).float()[None] / 255,
                        rs800=to_t(im.crop((0, 0, 800, h)))[None], gs800=to_t(im.crop((800, 0, w, h)))[None]))
    return out
test_set = load_samples(real_paths)
s0 = test_set[0]; print({k: tuple(v.shape) for k, v in s0.items() if torch.is_tensor(v)})

ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
ema_sd = {k[len('ema_model.'):]: v for k, v in ck['ema'].items() if k.startswith('ema_model.')}; del ck; gc.collect()
def build(steps, size=None):
    m = Unet(dim=cfg['model']['dim'], dim_mults=tuple(cfg['model']['dim_mults']), num_classes=cfg['model']['num_classes'], cond_drop_prob=cfg['model']['cond_drop_prob'])
    d = RS_Diffusion(m, image_size=size or cfg['diffusion']['image_size'], timesteps=cfg['diffusion']['timesteps'], sampling_timesteps=steps,
                     beta_schedule=cfg['diffusion']['beta_schedule'], objective=cfg['diffusion']['objective'])
    r = d.load_state_dict(ema_sd, strict=False)
    if r.missing_keys or r.unexpected_keys: print('load_state_dict: missing', r.missing_keys[:3], 'unexpected', r.unexpected_keys[:3])
    return d.cuda().eval()

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/kaggle/working/RS-Diffusion/model/diffusion_model.py:831: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)


{'flow': (1, 2, 256, 256), 'rs': (1, 3, 64, 64), 'rs_hi': (1, 3, 256, 256), 'gs_hi': (1, 3, 256, 256), 'rs800': (1, 3, 600, 800), 'gs800': (1, 3, 600, 800)}


## 3. The post-processing variants (the boxes of the diagram)

In [3]:
from skimage.metrics import peak_signal_noise_ratio as psnr_fn, structural_similarity as ssim_fn
from torch.utils.flop_counter import FlopCounterMode

def up_flow(f, hw, mode='bilinear', rate=True):
    # "resize + rescale": resize the flow field to (h, w) and multiply dx by w/w0 and dy by h/h0 (same as the repo's upsample2d_flow_as for bilinear)
    h, w = hw; _, _, h0, w0 = f.shape
    r = F.interpolate(f, size=(h, w), mode=mode, **({} if mode == 'nearest' else dict(align_corners=True)))
    return torch.cat([r[:, :1] * (w / w0), r[:, 1:] * (h / h0)], 1) if rate else r

def to01(x):
    x = x.detach().float().cpu()[0]
    if x.min() < 0: x = (x + 1) / 2
    return x.clamp(0, 1).permute(1, 2, 0).numpy()

def pipe_lowres(f, s, hw, rs_key):                       # A: warp the 64x64 image with the 64x64 flow, then upscale the corrected image
    w = flow_warp(s['rs'].cuda(), f, pad='zeros', mode='bilinear')
    return F.interpolate(w, size=hw, mode='bicubic', align_corners=False).clamp(0, 1), up_flow(f, hw)
def pipe_full(mode, rate):                               # B-E: upsample the flow, warp the full-resolution image
    def fn(f, s, hw, rs_key):
        fu = up_flow(f, hw, mode, rate)
        return flow_warp(s[rs_key].cuda(), fu, pad='zeros', mode='bilinear'), fu
    return fn
PIPES = {'A low-res warp + image upscale': pipe_lowres,
         'B full-res warp, flow bilinear+rescale (diagram)': pipe_full('bilinear', True),
         'C full-res warp, flow bicubic+rescale': pipe_full('bicubic', True),
         'D full-res warp, flow nearest+rescale': pipe_full('nearest', True),
         'E full-res warp, bilinear, NO rescale': pipe_full('bilinear', False)}
RES = {'256x256': dict(hw=(256, 256), rs='rs_hi', gs='gs_hi'), '800x600': dict(hw=(600, 800), rs='rs800', gs='gs800')}

def timed(fn):
    torch.cuda.synchronize(); t = time.perf_counter(); out = fn(); torch.cuda.synchronize(); return out, (time.perf_counter() - t) * 1000

# sanity: my up_flow equals the repo's function for the diagram's setting
_f = torch.randn(1, 2, 64, 64).cuda(); _ref = torch.zeros(1, 3, 256, 256).cuda()
print('up_flow vs repo upsample2d_flow_as, max abs diff:', (up_flow(_f, (256, 256)) - upsample2d_flow_as(_f, _ref, mode='bilinear', if_rate=True)).abs().max().item())

def gflops(fn):
    try:
        with FlopCounterMode(display=False) as fc: fn()
        return fc.get_total_flops() / 1e9
    except Exception as e:
        print('flop count failed:', repr(e)[:80]); return float('nan')

up_flow vs repo upsample2d_flow_as, max abs diff: 0.0


## 4. Run: predict the flow once per sample, then apply every variant to that same flow

In [4]:
rows, model_info = [], {}
cls = torch.zeros(1, dtype=torch.int64, device='cuda')

@torch.inference_mode()
def get_flows(diff, samples):
    sm = lambda rs: diff.sample(classes=cls, condition=rs, cond_scale=1).float()
    for s in samples[:N_WARMUP]: sm(s['rs'].cuda())
    flows, lat = [], []
    for i, s in enumerate(samples):
        torch.manual_seed(i); f, dt = timed(lambda: sm(s['rs'].cuda())); flows.append(f); lat.append(dt)
    return flows, lat, gflops(lambda: sm(samples[0]['rs'].cuda()))

@torch.inference_mode()
def apply_variants(flows, steps):
    for r, cfg_r in RES.items():
        hw = cfg_r['hw']
        for s, f in zip(test_set, flows):
            gs = to01(s[cfg_r['gs']]); true_up = up_flow(s['flow'].cuda(), hw)             # true flow at the evaluation resolution
            rs01 = to01(s[cfg_r['rs']])
            rows.append(dict(res=r, steps=steps, variant='0 no correction', sample=s['name'], psnr=psnr_fn(gs, rs01, data_range=1),
                             ssim=ssim_fn(gs, rs01, data_range=1, channel_axis=2), epe=float('nan'), post_ms=0.0))
            for name, pipe in PIPES.items():
                (img, fu), ms = timed(lambda: pipe(f, s, hw, cfg_r['rs']))
                pred = to01(img)
                rows.append(dict(res=r, steps=steps, variant=name, sample=s['name'], psnr=psnr_fn(gs, pred, data_range=1),
                                 ssim=ssim_fn(gs, pred, data_range=1, channel_axis=2), epe=float((fu - true_up).pow(2).sum(1).sqrt().mean()), post_ms=ms))

for st in STEPS_LIST:
    d = build(st); flows, lat, gf = get_flows(d, test_set)
    model_info[st] = dict(model_ms=float(np.median(lat)), gflops=gf, params_m=sum(p.numel() for p in d.model.parameters()) / 1e6)
    del d; gc.collect(); torch.cuda.empty_cache()
    apply_variants(flows, st); print('done: steps =', st, model_info[st])

sampling loop time step:   0%|          | 0/8 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/8 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/8 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/8 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/8 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/8 [00:00<?, ?it/s]

... (100 outputs left out)

sampling loop time step:   0%|          | 0/1 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/1 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/1 [00:00<?, ?it/s]

done: steps = 1 {'model_ms': 33.27122300004248, 'gflops': 36.912480256, 'params_m': 79.882731}


## 5. Results (dPSNR = paired difference vs variant B, the diagram's design, with 95% CI)

In [5]:
res = pd.DataFrame(rows)
ref = res[res.variant.str.startswith('B ')].set_index(['res', 'steps', 'sample'])[['psnr', 'ssim']].add_suffix('_B')
res = res.merge(ref, left_on=['res', 'steps', 'sample'], right_index=True); res['dpsnr'] = res.psnr - res.psnr_B
ci = lambda x: 1.96 * x.std() / np.sqrt(len(x))
summ = res.groupby(['res', 'steps', 'variant'], sort=False).agg(psnr=('psnr', 'mean'), dpsnr=('dpsnr', 'mean'), dpsnr_ci95=('dpsnr', ci),
                                                                ssim=('ssim', 'mean'), epe_px=('epe', 'mean'), post_ms=('post_ms', 'median')).reset_index()
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 20)
for r in RES:
    for st in STEPS_LIST:
        print(f'===== evaluated at {r} | RS-Diffusion {st} steps | model latency {model_info[st]["model_ms"]:.1f} ms | {model_info[st]["gflops"]:.1f} GFLOPs per image =====')
        print(summ[(summ.res == r) & (summ.steps == st)].drop(columns=['res', 'steps']).round(3).to_string(index=False)); print()
res.to_csv('/kaggle/working/flowdesign_results_per_sample.csv', index=False); summ.to_csv('/kaggle/working/flowdesign_results.csv', index=False)

===== evaluated at 256x256 | RS-Diffusion 8 steps | model latency 183.2 ms | 295.3 GFLOPs per image =====
                                         variant   psnr   dpsnr  dpsnr_ci95  ssim  epe_px  post_ms
                                 0 no correction 15.203 -10.958       1.129 0.430     NaN    0.000
                  A low-res warp + image upscale 21.470  -4.691       0.823 0.615   0.662    0.805
B full-res warp, flow bilinear+rescale (diagram) 26.161   0.000       0.000 0.860   0.662    1.256
           C full-res warp, flow bicubic+rescale 26.072  -0.089       0.032 0.860   0.666    1.270
           D full-res warp, flow nearest+rescale 25.806  -0.355       0.109 0.857   0.678    1.238
           E full-res warp, bilinear, NO rescale 16.466  -9.695       1.126 0.485   5.953    1.174

===== evaluated at 256x256 | RS-Diffusion 1 steps | model latency 33.3 ms | 36.9 GFLOPs per image =====
                                         variant   psnr   dpsnr  dpsnr_ci95  ssim  epe_px  post_

## 6. Optional: does a higher-resolution flow come for free? (diffusion model at a larger input, no retraining)
The pretrained model was trained at 64x64. This feeds it a larger input (default 128x128) **without retraining** and warps the 256x256 image with its flow. Expect higher cost; if quality also drops, that supports predicting flow at low resolution. If the architecture cannot take another size, the cell prints why and moves on.

In [6]:
if RUN_HIGHRES:
    for sz in HIGHRES:
        try:
            d = build(STEPS_LIST[0], size=sz); hr = []
            def run(s, i):
                rs = F.interpolate(s['rs_hi'].cuda(), size=(sz, sz), mode='bilinear', antialias=True, align_corners=False)
                torch.manual_seed(i); return d.sample(classes=cls, condition=rs, cond_scale=1).float(), rs
            with torch.inference_mode():
                for s in test_set[:N_WARMUP]: run(s, 0)
                for i, s in enumerate(test_set):
                    (f, _), dt = timed(lambda: run(s, i)); fu = up_flow(f, (256, 256)); gs = to01(s['gs_hi'])
                    pred = to01(flow_warp(s['rs_hi'].cuda(), fu, pad='zeros', mode='bilinear'))
                    hr.append(dict(psnr=psnr_fn(gs, pred, data_range=1), ssim=ssim_fn(gs, pred, data_range=1, channel_axis=2),
                                   epe=float((fu - up_flow(s['flow'].cuda(), (256, 256))).pow(2).sum(1).sqrt().mean()), ms=dt))
                gf = gflops(lambda: run(test_set[0], 0))
            hr = pd.DataFrame(hr); base = summ[(summ.res == '256x256') & (summ.steps == STEPS_LIST[0]) & summ.variant.str.startswith('B ')].iloc[0]
            print(f'diffusion at {sz}x{sz} input (pretrained at 64, no retraining), {STEPS_LIST[0]} steps -> flow upsampled to 256 and applied to the full-res image')
            print(f'  PSNR {hr.psnr.mean():.2f} vs {base.psnr:.2f} (64x64) | SSIM {hr.ssim.mean():.3f} vs {base.ssim:.3f} | EPE {hr.epe.mean():.3f} vs {base.epe_px:.3f} | '
                  f'latency {hr.ms.median():.0f} ms vs {model_info[STEPS_LIST[0]]["model_ms"]:.0f} ms | {gf:.0f} vs {model_info[STEPS_LIST[0]]["gflops"]:.0f} GFLOPs')
            del d; gc.collect(); torch.cuda.empty_cache()
        except Exception as e:
            print(f'could not run the diffusion model at {sz}x{sz}: {repr(e)[:300]}')

sampling loop time step:   0%|          | 0/8 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/8 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/8 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/8 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/8 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/8 [00:00<?, ?it/s]

... (45 outputs left out)

sampling loop time step:   0%|          | 0/8 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/8 [00:00<?, ?it/s]

sampling loop time step:   0%|          | 0/8 [00:00<?, ?it/s]

diffusion at 128x128 input (pretrained at 64, no retraining), 8 steps -> flow upsampled to 256 and applied to the full-res image
  PSNR 17.01 vs 26.16 (64x64) | SSIM 0.506 vs 0.860 | EPE 4.522 vs 0.662 | latency 398 ms vs 183 ms | 1199 vs 295 GFLOPs
